# 02 — Composition-group train, validation and test splits
Generate domain-specific splits from notebook 01's feature table. **Input:** `dataset_features.csv`. **Outputs:** `dataset_splits_folds_N.csv`, `_O.csv` and `_M.csv` under the selected project's `datasets/` directory.

Use the same project as in notebook 01. Skip this notebook when reproducing PAPER_V2 from archived splits: rerunning it replaces split files. Domain N denotes inorganic materials and is displayed as I in figures; symmetry label N independently denotes the positive, noncentrosymmetric class.


In [ ]:
from pathlib import Path
import sys
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "project_config.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from project_config import make_cfg
from private_paths import COD_DIR, MP_DIR
from pipelines.folds_pipeline import FoldsPipeline, FoldsPipelineConfig
PROJECT_NAME = "COD_CANON_V2"
cfg = make_cfg(project_name=PROJECT_NAME,
               root_cif={"COD_CANON_V2": COD_DIR, "MP_ALL": MP_DIR}[PROJECT_NAME])
FEATURE_STEPS = ("magpie",)
DOMAINS = ("N", "O", "M")
fold_cfg = FoldsPipelineConfig(group_col="group_id", domains=DOMAINS,
    outer_n_splits=5, inner_n_splits=5, seed=42, test_outer_fold=0,
    min_class_samples=20)


## Split protocol
Five shuffled outer partitions are generated with `StratifiedGroupKFold` and seed 42. Outer fold 0 is the fixed test set. The remaining records receive five inner validation folds with the same seed. This is one held-out test set with five-fold inner CV, rather than five outer model evaluations.

Composition `group_id` values remain together within a domain. Invalid labels and missing space-group numbers are filtered before splitting. Group validation during preparation and split validation describe different stages; final counts must be read from the resulting split tables.


In [ ]:
folds = FoldsPipeline(cfg, fold_cfg)
dfs_by_domain = folds.run(steps=FEATURE_STEPS)


## Output interpretation
`outer_fold` records the outer assignment. Test rows have `is_test=1` and `fold=-1`; non-test rows have validation fold IDs 0–4. The tables retain identifiers, formulas, targets, composition groups and the 132 Magpie columns required by notebook 03. Group sizes make the record-level test fraction approximately 20%.
